In [43]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,OneHotEncoder,StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression,Lasso,Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [44]:
df=pd.read_csv('cardekho_3.csv',index_col=0)
df.head()

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [45]:
dict1={
    "name":"kishor",
    "age":12
}
print(len(list(dict1)))

2


Feature Engineering
# Data Cleaning

In [46]:
df.isnull().sum()

car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

remove unnecessary columns

In [47]:
df.drop('car_name',axis=1,inplace=True)
df.drop('brand',axis=1,inplace=True)

In [48]:
df.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [49]:
num_features=[feature for feature in df.columns if df[feature].dtype != 'O']
print(num_features)
cat_features=[feature for feature in df.columns if df[feature].dtype == 'O']
print(cat_features)
discrete_features=[feature for feature in num_features if len(df[feature].unique())<=25]
print(discrete_features)
continuous_features=[feature for feature in num_features if feature not in discrete_features]
print(continuous_features)

['model', 'vehicle_age', 'km_driven', 'seller_type', 'fuel_type', 'transmission_type', 'mileage', 'engine', 'max_power', 'seats', 'selling_price']
[]
['vehicle_age', 'seller_type', 'fuel_type', 'transmission_type', 'seats']
['model', 'km_driven', 'mileage', 'engine', 'max_power', 'selling_price']


In [50]:
X=df.drop('selling_price',axis=1)
Y=df['selling_price']

In [51]:
X.shape,Y.shape

((15411, 10), (15411,))

# Encoding and scaling

In [52]:
df['model'].unique(),df['model'].value_counts()

(<StringArray>
 [        'Alto',        'Grand',          'i20',     'Ecosport',
       'Wagon R',          'i10',        'Venue',        'Swift',
         'Verna',       'Duster',
  ...
      'Panamera',      'Alturas',       'Altroz',           'NX',
      'Carnival',            'C',           'RX',        'Ghost',
  'Quattroporte',       'Gurkha']
 Length: 120, dtype: str,
 model
 i20             906
 Swift Dzire     890
 Swift           781
 Alto            778
 City            757
                ... 
 Altroz            1
 C                 1
 Ghost             1
 Quattroporte      1
 Gurkha            1
 Name: count, Length: 120, dtype: int64)

In [53]:
le=LabelEncoder()
X['model']=le.fit_transform(X['model'])

In [54]:
X.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,7,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,54,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,118,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,7,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,38,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


In [55]:
len(df['seller_type'].unique()),len(df['fuel_type'].unique()),len(df['transmission_type'].unique())

(3, 5, 2)

In [56]:
onehot_columns=['seller_type','fuel_type','transmission_type']
num_features=X.select_dtypes(include='number').columns.tolist()
X[onehot_columns]=X[onehot_columns].astype(object)

In [57]:
numeric_transfer=StandardScaler()
oh_transfer=OneHotEncoder(drop='first')

In [58]:
preprocessing=ColumnTransformer(
    [
        ('OneHotEncoder',oh_transfer,onehot_columns),
        ('StandardScaler',numeric_transfer,num_features)
    ],remainder='passthrough'
)

In [59]:
X=preprocessing.fit_transform(X)

In [60]:
X

array([[ 1.        ,  0.        ,  0.        , ..., -1.32425883,
        -1.26335238, -0.40302241],
       [ 1.        ,  0.        ,  0.        , ..., -0.55471774,
        -0.43257082, -0.40302241],
       [ 1.        ,  0.        ,  0.        , ..., -0.55471774,
        -0.47911321, -0.40302241],
       ...,
       [ 0.        ,  0.        ,  1.        , ...,  0.02291783,
         0.06822523, -0.40302241],
       [ 0.        ,  0.        ,  1.        , ...,  1.32979434,
         0.91715831,  2.07344426],
       [ 0.        ,  0.        ,  0.        , ...,  0.02099878,
         0.39588361, -0.40302241]], shape=(15411, 14))

In [61]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

In [62]:
def evaluateModel(true,predicted):
    print(f"mean aboslute error : {mean_absolute_error(true,predicted)}")
    print(f"mean sqaured error : {mean_squared_error(true,predicted)}")
    print(f"r2 Score : {r2_score(true,predicted)}")

In [63]:
models={
    'LinearRegression':LinearRegression(),
    'Lasso':Lasso(),
    'Ridge':Ridge(),
    'KNN':KNeighborsRegressor(),
    'Decision Tree':DecisionTreeRegressor(),
    'Random Forest':RandomForestRegressor()
}

In [64]:
for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(X_train,Y_train)
    Y_train_pred=model.predict(X_train)
    Y_test_pred=model.predict(X_test)
    print("\n",list(models.keys())[i])
    evaluateModel(Y_train,Y_train_pred)
    evaluateModel(Y_test,Y_test_pred)


 LinearRegression
mean aboslute error : 268101.6070829936
mean sqaured error : 306756099359.7596
r2 Score : 0.6217719576765959
mean aboslute error : 279618.5794158427
mean sqaured error : 252550062888.56552
r2 Score : 0.6645109298852006

 Lasso
mean aboslute error : 268099.2226498116
mean sqaured error : 306756104248.3742
r2 Score : 0.6217719516489697
mean aboslute error : 279614.74610341265
mean sqaured error : 252549134806.78134
r2 Score : 0.6645121627547996

 Ridge
mean aboslute error : 268059.8014688311
mean sqaured error : 306756818740.9266
r2 Score : 0.6217710706848424
mean aboslute error : 279557.2168930274
mean sqaured error : 252540243247.96854
r2 Score : 0.6645239743566811

 KNN
mean aboslute error : 91425.63270603504
mean sqaured error : 106193229526.28163
r2 Score : 0.8690645193508602
mean aboslute error : 112526.34609146934
mean sqaured error : 64021344520.150826
r2 Score : 0.9149536488136147

 Decision Tree
mean aboslute error : 5164.819922128488
mean sqaured error : 432

task do hyper parameter tuning